# Day3_03D_CrewAI_Custom_Tools

## CrewAI - Giving Agents Tools

**Duration:** 20 minutes  
**Audience:** Engineering Faculty  
**Environment:** VS Code Notebook  
**Session:** Day 3 - CrewAI  

---

## Learning Objectives

By the end of this notebook, participants will be able to:

- Understand why CrewAI Agents need tools.
- Explain the difference between Agent knowledge and Agent capability.
- Create a simple custom tool.
- Attach tools to CrewAI Agents.
- Build a DSU-style Agent that uses tools to answer questions.
- Relate CrewAI tools to enterprise integrations such as APIs, databases, and internal systems.

---

## Previous Notebook Recap

In the previous CrewAI notebooks, we learned:

- How to create Agents
- How to create Tasks
- How to create Crews
- How task context chaining works
- How sequential and hierarchical processes differ

Now we will learn the fourth major CrewAI concept:

> Tool

In CrewAI:

```text
Agent = Who does the work
Task = What needs doing
Tool = What the Agent can use
Crew = How Agents collaborate
```


# 1. Why Do Agents Need Tools?

An LLM-based Agent can reason and write.

But by itself, it cannot directly:

- Check a database
- Read live college information
- Calculate using a controlled function
- Call an API
- Search internal documents
- Fetch student support information

Tools give Agents external capability.

---

## Simple Analogy

A faculty member has knowledge.

But to conduct class effectively, the faculty member may use:

- Projector
- Attendance system
- LMS
- Lab computer
- Library portal

Similarly, an Agent uses tools.


# 2. Knowledge vs Capability

| Without Tools | With Tools |
|---|---|
| Agent answers from model knowledge | Agent can call functions |
| May guess facts | Can use controlled data |
| Limited to text reasoning | Can perform actions |
| Not connected to systems | Can connect to APIs/databases |
| Good for explanation | Better for enterprise workflows |

---

## Teaching Point

Tools are what make Agentic AI practical.

Without tools, an Agent is mostly a smart text generator.

With tools, an Agent can interact with systems.


# 3. Architecture for This Notebook

We will create a simple DSU Faculty Support Agent.

The Agent will use tools to answer:

- Library timing
- Department contact
- FDP session duration
- Simple calculator-style query

```text
Faculty Question
   ↓
CrewAI Agent
   ↓
Custom Tool
   ↓
Controlled Answer
```

This keeps the demo simple and classroom-friendly.


# 4. Setup

Install dependencies if needed:

```bash
pip install crewai crewai-tools python-dotenv
```

The `.env` file should be available in the project root.


In [ ]:
from pathlib import Path
from dotenv import load_dotenv
import os

from crewai import Agent, Task, Crew, Process
from crewai.tools import tool

# 5. Load Environment Variables

We will use the standard FDP root `.env` loading pattern.


In [ ]:
current = Path.cwd().resolve()

for folder in [current] + list(current.parents):
    env = folder / ".env"
    if env.exists():
        load_dotenv(env)
        break

print("API Key Loaded:", os.getenv("OPENAI_API_KEY") is not None)

# 6. Create Our First Custom Tool

CrewAI allows us to create tools using the `@tool` decorator.

A tool is basically a Python function that the Agent can use.

Important parts:

- Tool name
- Function input
- Function output
- Clear description


In [ ]:
@tool("DSU Library Timing Tool")
def dsu_library_timing_tool(query: str) -> str:
    """
    Returns DSU-style library timing information.
    Use this tool when the user asks about library opening hours or availability.
    """

    return (
        "The DSU central library is generally available from 9:00 AM to 6:00 PM "
        "on working days. For exact holiday or exam-period timings, faculty should "
        "confirm with the library office."
    )

## What did we just learn?

We created a tool.

The tool has:

- A name
- A description
- Python logic
- A return value

Now the Agent can use this controlled function instead of guessing.


# 7. Create More Simple Tools

Let us create two more DSU-style tools.

These are not connected to real systems.

They are classroom-safe mock tools to explain the concept.


In [ ]:
@tool("DSU Department Contact Tool")
def dsu_department_contact_tool(department: str) -> str:
    """
    Returns sample department contact information.
    Use this when the user asks for department contact or coordinator details.
    """

    contacts = {
        "aiml": "AI&ML Department Coordinator: Prof. Meena. Email: aiml.office@example.edu",
        "cse": "CSE Department Coordinator: Prof. Kumar. Email: cse.office@example.edu",
        "ece": "ECE Department Coordinator: Prof. Divya. Email: ece.office@example.edu",
    }

    key = department.lower().replace("&", "").replace(" ", "")

    if "ai" in key or "ml" in key:
        return contacts["aiml"]
    elif "cse" in key:
        return contacts["cse"]
    elif "ece" in key:
        return contacts["ece"]
    else:
        return "Department contact not found. Please check with the academic office."


@tool("FDP Duration Calculator Tool")
def fdp_duration_calculator_tool(days: int) -> str:
    """
    Calculates total FDP duration assuming 6 hours per day.
    Use this when the user asks about total FDP hours.
    """

    total_hours = days * 6
    return f"For a {days}-day FDP, the total duration is approximately {total_hours} hours." 

# 8. Create a Tool-Using Agent

Now we create an Agent and attach the tools.

Notice the `tools=[...]` argument.


In [ ]:
faculty_support_agent = Agent(
    role="DSU Faculty Support Assistant",
    goal="Answer faculty support questions using available tools whenever required",
    backstory="""
    You are a helpful faculty support assistant for an engineering college.
    You provide simple, practical answers. When a question requires college-specific
    information or calculation, you use the available tools instead of guessing.
    """,
    tools=[
        dsu_library_timing_tool,
        dsu_department_contact_tool,
        fdp_duration_calculator_tool,
    ],
    verbose=True,
    allow_delegation=False,
)

# 9. Create a Task That Requires Tool Usage

The task should clearly ask the Agent to use tools where relevant.


In [ ]:
faculty_query_task = Task(
    description="""
    Answer the following faculty question:

    "What are the library timings, who is the AI&ML department contact,
    and how many hours will a 5-day FDP have?"

    Use tools wherever needed. Keep the answer concise and faculty-friendly.
    """,
    expected_output="A concise answer using available tool information.",
    agent=faculty_support_agent,
)

# 10. Create and Run the Crew

This is a single-Agent Crew.

That is fine.

A Crew does not always need many Agents.

Sometimes one Agent with useful tools is enough.


In [ ]:
tool_crew = Crew(
    agents=[faculty_support_agent],
    tasks=[faculty_query_task],
    process=Process.sequential,
    verbose=True,
)

result = tool_crew.kickoff()

print(result)

# 11. What Did We Just Build?

We built a CrewAI Agent that can use tools.

```text
Question
   ↓
Agent decides what is needed
   ↓
Tool is called
   ↓
Tool result is used
   ↓
Final answer
```

This is similar to the Function Tools concept we learned in OpenAI SDK.

The framework syntax is different, but the idea is the same.


# 12. Modify the Query

Try a different faculty question.

This makes the demo interactive.


In [ ]:
faculty_query_task.description = """
Answer the following faculty question:

"Who should I contact in the CSE department and what is the total duration of a 3-day FDP?"

Use tools wherever needed. Keep the answer concise.
"""

result = tool_crew.kickoff()

print(result)

# 13. Tool Design Guidelines

Good tools are:

- Simple
- Specific
- Clearly named
- Well described
- Reliable
- Easy to test independently

---

## Poor Tool

```python
@tool("Info Tool")
def info(x):
    ...
```

This is vague.

---

## Better Tool

```python
@tool("FDP Duration Calculator Tool")
def fdp_duration_calculator_tool(days: int) -> str:
    ...
```

This tells the Agent exactly when to use it.


# 14. Test Tools Directly

Before connecting a tool to an Agent, always test it directly.

This is a good classroom habit.


In [ ]:
print(dsu_library_timing_tool.run("library timing"))
print(dsu_department_contact_tool.run("AI&ML"))
print(fdp_duration_calculator_tool.run(5))

# 15. Enterprise Connection

In real enterprise systems, CrewAI tools may connect to:

- CRM systems
- HRMS systems
- Ticketing tools
- Databases
- REST APIs
- Knowledge bases
- Monitoring systems
- DevOps platforms

---

## Banking Example

```text
Customer Query
   ↓
Agent
   ├── Account Lookup Tool
   ├── Transaction Status Tool
   ├── Loan Eligibility Tool
   └── Complaint Ticket Tool
   ↓
Customer Response
```

Tools make Agents enterprise-ready.


# 16. Academic Example

In an engineering college, tools may connect to:

- LMS
- Attendance system
- Library portal
- Exam timetable system
- Placement database
- Course catalogue
- Faculty directory

---

## Example

```text
Student Question
   ↓
College Assistant Agent
   ├── Timetable Tool
   ├── Faculty Directory Tool
   ├── Assignment Deadline Tool
   └── Library Timing Tool
   ↓
Student Response
```


# 17. Safety Note

Tools make Agents powerful.

But they also increase risk.

If a tool can:

- update data
- send email
- make payment
- delete records
- approve requests

then you need:

- guardrails
- approval workflow
- access control
- logging
- human review
- audit trail

For this FDP, we start with safe read-only tools.


# 18. Classroom Exercise

Create a new tool:

```python
@tool("Course Outcome Tool")
def course_outcome_tool(course_name: str) -> str:
    ...
```

It should return simple course outcomes for:

- Agentic AI
- Machine Learning
- MLOps

Then create an Agent that uses this tool to answer faculty questions.


In [ ]:
# Exercise Starter Code

@tool("Course Outcome Tool")
def course_outcome_tool(course_name: str) -> str:
    """
    TODO:
    Return course outcomes for a given course.
    """

    # TODO:
    # Create a dictionary of course outcomes.
    # Return matching course outcome.
    return "TODO: return course outcome"


# TODO:
# 1. Create a Course Advisor Agent.
# 2. Attach course_outcome_tool.
# 3. Create a task asking about Agentic AI course outcomes.
# 4. Create a Crew and run kickoff().


# 19. Challenge Exercise

Create a simple **Placement Readiness Crew** with tools.

Tools:

1. Resume Tip Tool
2. Interview Question Tool
3. HR Preparation Tool

Agent:

- Placement Preparation Assistant

Task:

> Help a final-year student prepare for a campus interview.

Expected output:

- Resume tips
- Technical interview preparation
- HR interview preparation
- Final 3-day preparation plan


# 20. Common Mistakes

Avoid these mistakes when using tools:

1. Tool name is too vague.
2. Tool description is unclear.
3. Tool returns too much text.
4. Tool input type is confusing.
5. Tool does too many things.
6. Tool is not tested separately.
7. Agent is not instructed to use tools.
8. Tool performs risky action without guardrails.

---

## Teaching Tip

For beginners, start with read-only tools.

Then later explain write/action tools with safety controls.


# 21. Key Takeaways

In this notebook, we learned:

- Tools give CrewAI Agents external capability.
- A tool is usually a Python function exposed to the Agent.
- Agents can decide when to use tools.
- Tool names and descriptions matter.
- Tools should be simple, reliable, and easy to test.
- Enterprise Agents often depend on tools to connect with systems.
- Safe tool design requires guardrails and logging.

---

## Final Mental Model

```text
Agent knows how to reason.
Tool helps the Agent act.
Crew organizes collaboration.
```

Tools are the bridge between AI reasoning and real-world systems.
